# Smart-Money Wallet Ranking — Hyperliquid

Ranks Hyperliquid wallets by a **risk-adjusted composite score** instead of the naive
"sort by 30-day PnL" (which just rewards account size and floods the top with passive
holders). Run top-to-bottom to produce the top-100 cohort. **All formulas and methodology are
in the final cell** — this notebook is meant to be a reference spec for implementation.

## The two-stage pipeline

The full leaderboard is ~40,000 wallets. We can't make a `portfolio` API call for each, so
Stage 1 filters cheaply (free leaderboard fields only), then Stage 2 spends the expensive
calls on a shortlist:

```
  ~40,000 leaderboard rows                 (1 free GET: pnl / roi / vlm + accountValue per wallet)
        │
        │  STAGE 1  — free eligibility filters, then sort
        │     drop sentinel rows (data quality);
        │     keep accountValue > 0  AND  month volume > 0  (active = traded >= once in 30d);
        │     sort by 30-day PnL, take the top SHORTLIST_SIZE
        ▼
  top ~500 candidates
        │
        │  STAGE 2  — expensive metrics (1 portfolio call each)
        │     build the equity curve -> Sharpe, profit factor, max drawdown, win-rate;
        │     require >= MIN_OBS daily points
        ▼
  active, scored wallets
        │  SMART SCORE = confidence x weighted percentile-rank composite
        ▼
  top COHORT_SIZE (100)  ← the smart-money cohort
```

Stage 1 is deliberately light — plain eligibility plus a PnL sort — so it's cheap and easy to
reason about. All of the *judgement* (risk-adjusted skill vs raw size) happens in the Stage-2
score, which re-ranks the 500 candidates.

## Data sources (no API key)

| Endpoint | Fields used | Stage |
|---|---|---|
| `GET stats-data.hyperliquid.xyz/Mainnet/leaderboard` | per wallet: `windowPerformances.month` → `pnl`, `roi`, `vlm`; `accountValue` | 1 (free, all wallets) |
| `POST api.hyperliquid.xyz/info {"type":"portfolio","user":addr}` | `month.accountValueHistory` (equity curve) + `month.pnlHistory` (**cumulative** PnL) | 2 (one call per shortlisted wallet) |

## Two design facts (validated live — keep in mind when implementing)

- **Profit factor is computed from the equity curve, not from fills.** Per-fill `closedPnl`
  reconstructs <2% of these wallets' real PnL (returns are mostly *unrealised* mark-to-market),
  so a fill-based profit factor would be wrong. The equity-curve version (gross up-day P&L ÷
  gross down-day P&L) is the Omega ratio at threshold 0 — same meaning, accurate for every wallet.
- **All risk metrics use the 30-day (`month`) window** because it is sampled ~daily (~31
  points). The `allTime` curve is capped at ~40–90 points regardless of account age (weekly /
  bi-weekly resolution), so it can't give a clean daily Sharpe — see the final cell for how to
  add a cheap full-history *track-record* signal instead.

In [1]:
# --- config -------------------------------------------------------------------------------
import time, requests
import numpy as np
import pandas as pd

HL_INFO_URL        = "https://api.hyperliquid.xyz/info"
HL_LEADERBOARD_URL = "https://stats-data.hyperliquid.xyz/Mainnet/leaderboard"

# ---- Stage 1: free eligibility + PnL sort over the whole leaderboard ----
SENTINEL_ALLTIME_PNL = -500.0   # HL placeholder for untracked-history wallets -> drop (data quality)
SHORTLIST_SIZE       = 500      # top-N by 30d PnL that advance to the expensive Stage 2

# ---- Stage 2: expensive per-wallet metrics + final score ----
MIN_OBS     = 10                # min daily equity points required to score a wallet
PF_CAP      = 5.0               # winsorize profit factor (>5 is already elite / a no-loss month)
CONF_FULL   = 21                # daily observations needed for full confidence weight
COHORT_SIZE = 100               # final cohort = top-N wallets by smart_score

# Smart-money weights (must sum to 1): 70% risk-adjusted skill/risk-control
# (Sharpe + max drawdown + profit factor + ROI), 30% scale (PnL + volume).
WEIGHTS = {"sharpe_30d": 0.25, "max_drawdown_30d": 0.15, "profit_factor_30d": 0.15,
           "roi_30d": 0.15, "pnl_30d_usd": 0.15, "volume_30d_usd": 0.15}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9


def hl_post(payload, max_attempts=5):
    """POST Hyperliquid /info with exponential backoff on HTTP 429."""
    delay = 2.0
    for _ in range(max_attempts):
        r = requests.post(HL_INFO_URL, json=payload, timeout=30)
        if r.status_code == 429:
            time.sleep(float(r.headers.get("Retry-After", delay)))
            delay = min(delay * 2, 30.0)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError("Hyperliquid /info still rate-limited after retries")


def _f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan

In [2]:
# --- STAGE 1: free eligibility filters + PnL sort ----------------------------------------
def stage1_shortlist(shortlist_size=SHORTLIST_SIZE):
    """Cut ~40k leaderboard wallets to the top `shortlist_size` by 30d PnL, using only the
    free leaderboard fields (one GET, no per-wallet calls).

    Filters (in order):
      1. drop the -500.0 all-time-PnL sentinel  -- HL's placeholder for untracked/migrated
         wallets (fake $200M+ PnL); ESSENTIAL because we sort by PnL, or they'd top the list.
      2. accountValue > 0                         -- must hold capital.
      3. month volume > 0                         -- traded at least once in the last 30d
                                                     (this also removes passive holders; a
                                                     separate trade-count filter isn't needed
                                                     and isn't available without paid fills calls).
    Then sort by 30d PnL and keep the top `shortlist_size`.
    """
    raw = requests.get(HL_LEADERBOARD_URL, timeout=60).json()
    rows = raw.get("leaderboardRows", raw if isinstance(raw, list) else [])

    def win(r, w):
        return dict(r.get("windowPerformances", [])).get(w, {})

    recs = []
    for r in rows:
        if _f(win(r, "allTime").get("pnl")) == SENTINEL_ALLTIME_PNL:   # 1. sentinel
            continue
        acc = _f(r.get("accountValue"))
        vlm = _f(win(r, "month").get("vlm"))
        if not (acc > 0):                                             # 2. zero account value
            continue
        if not (vlm > 0):                                            # 3. zero volume (inactive)
            continue
        recs.append({
            "trader_address":    r.get("ethAddress"),
            "account_value_usd": acc,
            "pnl_30d_usd":       _f(win(r, "month").get("pnl")),
            "roi_30d":           _f(win(r, "month").get("roi")),
            "volume_30d_usd":    vlm,
            "alltime_pnl_usd":   _f(win(r, "allTime").get("pnl")),    # kept for optional track-record
            "alltime_roi":       _f(win(r, "allTime").get("roi")),
        })

    df = pd.DataFrame(recs).sort_values("pnl_30d_usd", ascending=False).reset_index(drop=True)
    df.insert(0, "stage1_rank_pnl", df.index + 1)
    return df.head(shortlist_size)


shortlist = stage1_shortlist()
print(f"STAGE 1: {len(shortlist)} candidates (top {SHORTLIST_SIZE} by 30d PnL; "
      f"accountValue>0, volume>0, sentinel dropped)")
print(f"         30d PnL range: ${shortlist['pnl_30d_usd'].min():,.0f} .. "
      f"${shortlist['pnl_30d_usd'].max():,.0f}")
shortlist.head()

STAGE 1: 500 candidates (top 500 by 30d PnL; accountValue>0, volume>0, sentinel dropped)
         30d PnL range: $225,018 .. $12,584,040


,stage1_rank_pnl,trader_address,account_value_usd,pnl_30d_usd,roi_30d,volume_30d_usd,alltime_pnl_usd,alltime_roi
0,1,0x4e23288cee4960f9f962195c22948e4bc7ae20c3,3.110437e+07,1.258404e+07,0.679321,1.960469e+09,1.776792e+07,1.320933
1,2,0x8def9f50456c6c4e37fa5d3d57f108ed23992dae,1.012196e+08,1.074818e+07,0.118645,3.789079e+07,2.733710e+06,0.026133
2,3,0xe867fbdad3291530e41530301ecb77693850c78e,7.224655e+07,9.010909e+06,0.142497,1.329196e+08,4.866034e+07,2.063084
3,4,0xbdfa4f4492dd7b7cf211209c4791af8d52bf5c50,6.153732e+07,7.715566e+06,0.142559,3.738361e+05,1.026547e+08,9.199981
4,5,0x03b9a189e2480d1e4c3007080b29f362282130fa,6.126539e+07,7.342874e+06,0.136137,2.911318e+07,7.167796e+07,13.399650


In [3]:
# --- STAGE 2: per-wallet risk metrics from each candidate's equity curve -----------------
def portfolio_metrics(port, pf_cap=PF_CAP):
    """Sharpe, volatility, profit factor, max drawdown and win-rate over the trailing 30
    days, from a wallet's `portfolio` response (the `month` window).

    HL returns, per window, `accountValueHistory` (equity curve A_t) and `pnlHistory`
    (CUMULATIVE trading PnL C_t) -- each a list of [ms_timestamp, value_string], sampled
    ~daily (~31 points). Steps (formulae in the final cell):

      1. Resample the curve to a fixed 1-day grid (last observation per day).
      2. daily_pnl    = diff of CUMULATIVE C_t.  Using pnl (not account value) isolates
                        trading P&L: deposits/withdrawals move A_t but NOT C_t.
      3. daily_return = daily_pnl / previous day's account value.
      4. sharpe       = mean/std of daily returns, annualised by sqrt(365).
      5. profit_factor= gross positive daily_pnl / gross negative daily_pnl (equity-curve).
      6. max_drawdown = worst peak-to-trough of the equity curve.

    Returns {"n_obs": n} only when there is too little history to be trustworthy.
    """
    seg = dict(port).get("month", {})
    avh = seg.get("accountValueHistory", [])
    ph  = seg.get("pnlHistory", [])
    if len(avh) < 3:
        return {"n_obs": 0}

    curve = pd.DataFrame({
        "t":       pd.to_datetime([p[0] for p in avh], unit="ms"),
        "acc":     [float(p[1]) for p in avh],
        "cum_pnl": [float(p[1]) for p in ph],
    }).set_index("t").sort_index()

    daily = curve.resample("1D").last().dropna()
    daily["pnl"] = daily["cum_pnl"].diff()
    daily["ret"] = (daily["pnl"] / daily["acc"].shift(1).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

    ret = daily["ret"].dropna()
    n = len(ret)
    if n < 2:
        return {"n_obs": n}

    sd = ret.std(ddof=1)
    gross_up   = daily["pnl"][daily["pnl"] > 0].sum()
    gross_down = -daily["pnl"][daily["pnl"] < 0].sum()
    pf = (gross_up / gross_down) if gross_down > 0 else (pf_cap if gross_up > 0 else np.nan)
    eq = daily["acc"]

    return {
        "n_obs":              n,
        "sharpe_30d":         (ret.mean() / sd) * np.sqrt(365) if sd and sd > 0 else np.nan,
        "volatility_30d":     sd * np.sqrt(365) if sd else np.nan,
        "profit_factor_30d":  min(pf, pf_cap) if pd.notna(pf) else np.nan,
        "max_drawdown_30d":   float((eq / eq.cummax() - 1).min()),
        "win_rate_days_30d":  float((daily["pnl"].dropna() > 0).mean()),
    }


def build_metrics(shortlist_df, delay=0.08):
    """One `portfolio` call per shortlisted wallet; attach the metrics to the frame."""
    rows = []
    for i, addr in enumerate(shortlist_df["trader_address"]):
        try:
            rows.append(portfolio_metrics(hl_post({"type": "portfolio", "user": addr})))
        except Exception:
            rows.append({"n_obs": 0})
        time.sleep(delay)
        if (i + 1) % 50 == 0:
            print(f"  portfolio {i + 1}/{len(shortlist_df)}")
    return pd.concat([shortlist_df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

In [4]:
# --- SMART SCORE: percentile-ranked, weighted, confidence-shrunk composite ---------------
def smart_money_score(df, weights=WEIGHTS, min_obs=MIN_OBS, conf_full=CONF_FULL,
                      cohort_size=COHORT_SIZE):
    """Score & rank the Stage-2 candidates, then return the top `cohort_size`
    (math in the final cell).

        smart_score = confidence * sum_m  weight_m * percentile_rank(metric_m)

    Eligibility (Stage-2 gate): score a wallet only if it has enough equity history to trust
    its risk metrics -- n_obs >= min_obs. (Volume > 0 is already guaranteed by Stage 1.)
    """
    scored = df[df["n_obs"].fillna(0) >= min_obs].copy()

    # 1. cross-sectional percentile rank of each metric -> [0, 1] (robust to fat tails).
    #    max_drawdown_30d is stored as a NEGATIVE number (0 = none, -1 = wiped out), so a
    #    smaller drawdown ranks higher automatically -- no sign flip needed.
    for m in weights:
        scored[f"P_{m}"] = scored[m].rank(pct=True)

    # 2. weighted blend, then 3. shrink by confidence (thin history -> discounted)
    scored["composite"]   = sum(w * scored[f"P_{m}"] for m, w in weights.items())
    scored["confidence"]  = (scored["n_obs"] / conf_full).clip(0.3, 1.0)
    scored["smart_score"] = scored["composite"] * scored["confidence"]

    scored = scored.sort_values("smart_score", ascending=False).reset_index(drop=True)
    scored.insert(0, "rank", scored.index + 1)
    return scored.head(cohort_size)

In [5]:
# --- run the full pipeline ----------------------------------------------------------------
metrics = build_metrics(shortlist)          # STAGE 2: ~500 portfolio calls (~5-8 min)
cohort  = smart_money_score(metrics)        # score + take top COHORT_SIZE

n_active = int((metrics["n_obs"].fillna(0) >= MIN_OBS).sum())
print(f"\nSTAGE 2: {n_active} of {len(metrics)} candidates had enough history to score")
print(f"COHORT : top {len(cohort)} by smart_score\n")

cols = ["rank", "stage1_rank_pnl", "trader_address", "smart_score", "sharpe_30d",
        "profit_factor_30d", "roi_30d", "pnl_30d_usd", "volume_30d_usd",
        "max_drawdown_30d", "win_rate_days_30d", "n_obs"]
top = cohort.head(20)[cols].copy()
top["trader_address"] = top["trader_address"].str.slice(0, 12) + "..."
print("TOP 20 SMART-MONEY WALLETS  --  Hyperliquid, trailing 30 days")
print(top.to_string(index=False, formatters={
    "smart_score":       "{:.3f}".format,
    "sharpe_30d":        "{:.2f}".format,
    "profit_factor_30d": "{:.2f}".format,
    "roi_30d":           "{:.1%}".format,
    "pnl_30d_usd":       "${:,.0f}".format,
    "volume_30d_usd":    "${:,.0f}".format,
    "max_drawdown_30d":  "{:.1%}".format,
    "win_rate_days_30d": "{:.0%}".format,
}))

  portfolio 50/500
  portfolio 100/500
  portfolio 150/500
  portfolio 200/500
  portfolio 250/500
  portfolio 300/500
  portfolio 350/500
  portfolio 400/500
  portfolio 450/500
  portfolio 500/500

STAGE 2: 499 of 500 candidates had enough history to score
COHORT : top 100 by smart_score

TOP 20 SMART-MONEY WALLETS  --  Hyperliquid, trailing 30 days
 rank  stage1_rank_pnl  trader_address smart_score sharpe_30d profit_factor_30d roi_30d pnl_30d_usd volume_30d_usd max_drawdown_30d win_rate_days_30d  n_obs
    1                8 0x9e8b1e51c6...       0.942       6.99              2.79  128.1%  $5,875,717   $255,853,126           -12.6%               58%     31
    2               39 0xb1ec7febea...       0.904       5.30              2.27   59.2%  $2,216,609   $339,662,613           -10.1%               55%     31
    3                1 0x4e23288cee...       0.871       4.75              2.34   67.9% $12,584,040 $1,960,468,745           -17.0%               68%     31
    4             

## Formulas & methodology

Reference spec for implementation. Two stages: a free eligibility+sort prefilter, then
expensive per-wallet metrics and the composite score.

---

### STAGE 1 — free prefilter (leaderboard fields only)

Goal: cut ~40k wallets to a `SHORTLIST_SIZE` (500) candidate pool with **no** per-wallet API
calls. Keep it simple — eligibility filters, then a PnL sort. All judgement is deferred to
Stage 2.

**Filters** (drop wallet $w$ unless all hold):

$$\text{allTime.pnl}(w)\neq -500.0 \quad\wedge\quad \text{accountValue}(w) > 0 \quad\wedge\quad \text{volume}_{30d}(w) > 0$$

- The $-500.0$ sentinel is HL's placeholder for untracked/migrated wallets (fake $200M+
  PnL); it is dropped **first** because we sort by PnL and it would otherwise take the top slots.
- $\text{volume}_{30d} > 0$ means the wallet **traded at least once** in the window (no volume ⇒
  no trades), so a separate minimum-trade-count filter is redundant — and trade count isn't in
  the leaderboard anyway (it needs paid per-wallet `userFills` calls). The stronger "is this a
  real ongoing trader" test is the Stage-2 $n_{\text{obs}}\ge 10$ gate.

**Selection.** Sort survivors by $\text{pnl}_{30d}$ descending, keep the top `SHORTLIST_SIZE`.

---

### STAGE 2 — per-wallet metrics (one `portfolio` call each)

Let a wallet's equity curve be sampled ~daily over the trailing 30 days: account value $A_t$
and **cumulative** trading PnL $C_t$ at day $t=0,1,\dots,T$ ($T\!+\!1\approx31$ points).

**Daily PnL** — difference of the cumulative curve (isolates trading P&L; a deposit moves
$A_t$ but not $C_t$):

$$\Delta_t = C_t - C_{t-1}$$

**Daily return** — PnL over the capital deployed the previous day:

$$r_t = \frac{\Delta_t}{A_{t-1}}$$

**Sharpe ratio** — annualised; no risk-free term (crypto convention, rf = 0), $\sqrt{365}$
because returns are daily and crypto trades every day:

$$\text{Sharpe} = \frac{\bar r}{\sigma_r}\,\sqrt{365},\qquad
\bar r=\tfrac1n\textstyle\sum_t r_t,\qquad
\sigma_r=\sqrt{\tfrac{1}{n-1}\textstyle\sum_t (r_t-\bar r)^2}$$

**Profit factor** — gross gains ÷ gross losses from the equity curve (Omega ratio at
threshold 0), capped at $\text{PF}_{\max}=5$:

$$\text{PF} = \frac{\sum_{t:\,\Delta_t>0}\Delta_t}{\bigl|\sum_{t:\,\Delta_t<0}\Delta_t\bigr|}$$

**Max drawdown** — worst peak-to-trough of the equity curve. **Scored.** Stored as a
**negative** number ($0$ = no drawdown, $-1$ = wiped out), so a *smaller* drawdown gets a
higher percentile rank automatically — no sign flip needed. This is the term that penalises
near-blowup wallets (which can otherwise post a strong Sharpe on the recovery):

$$\text{MDD} = \min_{t}\!\left(\frac{A_t}{\max_{s\le t}A_s}-1\right)$$

**ROI, PnL, Volume** are read straight from the leaderboard's 30-day window (carried from Stage 1).

**Stage-2 gate.** Score a wallet only if $n_{\text{obs}}\ge 10$ (enough daily points for a
trustworthy standard deviation).

---

### SMART SCORE — the composite

The five scored metrics live on incomparable scales (volume in \$100M, ROI in %, Sharpe ~single
digits, PF ~1–3) and crypto is fat-tailed, so we **rank, not z-score**.

**1. Percentile rank.** For metric $m$ and wallet $i$ among the $N$ scored wallets:

$$P_m(i) = \frac{\bigl|\{\,j : x_m(j)\le x_m(i)\,\}\bigr|}{N}\in(0,1]$$

Scale-free, outlier-robust, interpretable (0.9 = top decile); negatives rank low naturally.

**2. Weighted composite** (weights sum to 1 ⇒ composite $\in(0,1]$):

$$\text{composite}(i)=\sum_{m} w_m\,P_m(i),\qquad
w=\big(\underbrace{0.25}_{\text{Sharpe}},\underbrace{0.15}_{\text{MaxDD}},\underbrace{0.15}_{\text{PF}},\underbrace{0.15}_{\text{ROI}},\underbrace{0.15}_{\text{PnL}},\underbrace{0.15}_{\text{Volume}}\big)$$

→ **70%** risk-adjusted skill & risk control (Sharpe + Max-DD + PF + ROI), **30%** scale
(PnL + Volume). Size breaks ties between equally-skilled traders but can't buy the top spot;
max drawdown pushes down wallets that nearly blew up.

**3. Confidence shrink** — damp thin histories so a tiny, noisy sample can't fluke to the top:

$$c(i)=\operatorname{clip}\!\left(\frac{n_i}{21},\,0.3,\,1.0\right)$$

**Final score** (rank descending, take top `COHORT_SIZE` = 100):

$$\boxed{\;\text{smart\_score}(i)=c(i)\cdot\text{composite}(i)\in(0,1]\;}$$

---

### Notes for implementation

- **Productionising**: land the raw daily equity curve into a `raw_hl_portfolio` table
  (`snapshot_date, trader_address, point_ts, account_value, cum_pnl`); derive the metrics in
  dbt (`int_wallet_returns`); store `smart_score` **and its five component percentiles as
  columns** on `fct_trader_daily` (keep ranking/LIMIT out of the warehouse — the client sorts).
  Put `WEIGHTS`, `SHORTLIST_SIZE`, `COHORT_SIZE` in `config.py`.
- **Tunables**: `SHORTLIST_SIZE` trades API load (one portfolio call each) against how deep
  down the PnL leaderboard you look; `COHORT_SIZE` is the final cohort size.
- **Optional full-history / track-record signal**: the leaderboard also carries `allTime.pnl`
  and `allTime.roi` for free (already loaded in Stage 1 as `alltime_pnl_usd` / `alltime_roi`).
  Add their percentile ranks as extra low-weight terms so a proven long-term trader isn't
  outranked by a one-month wonder. A daily-resolution full-history *Sharpe* is **not**
  available — the `allTime` equity curve is capped at ~40–90 points (weekly/bi-weekly).
- **Profit factor caveat**: equity-curve (daily) PF $\ne$ per-trade PF; it's the honest,
  computable version here because fill `closedPnl` is unreliable (see the intro). PF is
  collinear with Sharpe (same daily series); **max drawdown is included as a second,
  independent (tail-risk) axis** so near-blowup wallets are penalised. For an even sharper
  penalty you could replace PF with Calmar $=$ annualised return $/\,|\text{MDD}|$.
- **Catastrophic drawdowns**: percentile ranking sends the worst drawdowns to ~0, but if you
  want a hard cut, drop wallets with $\text{MDD}\le-95\%$ (near-liquidation / possibly a bad
  equity-curve point) before scoring.